# 03 · Results analysis

This notebook turns the saved runs into the **tables and figures of the paper**: mean ± sd per arm and budget, a significance test for every contrast, and the plots. It only reads the result folders; it does not train anything and needs no GPU. The code lives in `src/gainsplit/analysis.py`.

**Where it runs.** Locally (from the repo) or on Kaggle. On Kaggle add the results as inputs and set the folders in step 2. Scores in the tables are in percentage points (macro-F1 0.882 is shown as 88.2).

## 1. Get the code
Makes the `gainsplit` package importable. Locally it uses the `src/` folder of this repo; on Kaggle it fetches the repo from GitHub. Without this step the later cells cannot find the analysis functions.

In [ ]:
import os, subprocess, sys
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    REPO_DIR = "/tmp/GainSplit"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Ariffurrhmn/GainSplit.git", REPO_DIR], check=True)
else:
    REPO_DIR = str(Path.cwd().parent) if Path.cwd().name == "experiments" else str(Path.cwd())
sys.path.insert(0, f"{REPO_DIR}/src")

import pandas as pd
from gainsplit import analysis as an
pd.set_option("display.width", 200)

## 2. Choose the result folders
Lists the folders that contain a `runs/` subfolder (Pass 1, later Pass 2a, ...). All of them are pooled into one table. If the same arm, budget and seed appears twice, the last folder in the list wins and a warning is printed. Edit this list when new passes finish. On Kaggle too, the folders come from the GitHub clone, because every finished pass is committed to the repo; no Kaggle dataset is needed.

In [ ]:
if ON_KAGGLE:
    # the results are committed to the repo, so the clone from step 1 already has them
    RESULTS_DIRS = [f"{REPO_DIR}/results/pass1"]   # add f"{REPO_DIR}/results/pass2a" later
    OUT_DIR = "/kaggle/working/analysis"
else:
    RESULTS_DIRS = [f"{REPO_DIR}/results/pass1"]   # add f"{REPO_DIR}/results/pass2a" later
    OUT_DIR = f"{REPO_DIR}/results/analysis"

N_BOOT = 10000     # bootstrap resamples
RNG_SEED = 0       # fixed so the numbers are reproducible

## 3. Load every run
Reads each `result.json` and notes where its prediction files are. One row per run (arm, budget, seed, scores). The count table shows how many seeds each arm has at each budget: it should be 5 everywhere a run was planned. A gap here means a missing run, so check it before trusting any number below.

In [ ]:
runs = an.load_runs(*RESULTS_DIRS)
print(len(runs), "runs")
runs.groupby(["arm", "budget"]).size().unstack(fill_value=0)

## 4. Mean ± sd table
For each arm and budget: the mean test macro-F1 over the seeds and the standard deviation across seeds. This is the main results table. The sd shows how much a single run moves around because of the seed (which labeled items were drawn, initialisation).

In [ ]:
summ = an.summary_table(runs, split="test", metric="macro_f1")
wide = an.summary_wide(summ)
an.save_table(summ, OUT_DIR, "summary_test_macro_f1_long")
an.save_table(wide, OUT_DIR, "summary_test_macro_f1")
wide

## 5. Same table on the dev set
The dev set was used for choices such as the learning rate. Showing it next to the test table lets you check that the picture is the same and nothing was tuned on the test set.

In [ ]:
summ_dev = an.summary_table(runs, split="dev", metric="macro_f1")
an.save_table(summ_dev, OUT_DIR, "summary_dev_macro_f1_long")
an.save_table(an.summary_wide(summ_dev), OUT_DIR, "summary_dev_macro_f1")
an.summary_wide(summ_dev)

## 6. Significance of each contrast (paired bootstrap)
The differences between arms are small (a few tenths of a point), so comparing seed means is too noisy. Instead we use the fact that **every arm is scored on the same test headlines**. We draw 10,000 resamples of the test headlines (with replacement). In each resample, every seed of both arms is scored on the *same* resampled headlines, the macro-F1 difference is computed per seed, and then averaged over seeds. The spread of these 10,000 averages gives a 95% interval; the p-value is twice the smaller share of resamples with a difference ≤ 0 or ≥ 0. A contrast is clearly different from zero when its interval excludes 0. Only seeds that both arms have are used (`n_seeds`).

Contrasts: the Pass 1 ones (A3−A1, A4−A1, A5−A1, A5−A3, A5−A4), the Pass 2 ones, and the interaction A5−A4−A3+A1 (is GAN + helpers more or less than the two effects added together?). Contrasts with a missing arm are skipped. All contrasts are reported with no multiple-testing correction, which the paper states.

In [ ]:
contrasts = an.contrast_table(runs, split="test", metric="macro_f1", n=N_BOOT, rng_seed=RNG_SEED)
an.save_table(contrasts, OUT_DIR, "contrasts_test_macro_f1")
contrasts.round({"mean_diff": 2, "ci_low": 2, "ci_high": 2, "p": 4})

## 7. One contrast in detail
Runs a single comparison by hand so you can see the output: `mean_diff` is the difference on the full test set, `boot_mean` the average over resamples, `ci_low`/`ci_high` the 95% interval, `p` the two-sided p-value. Change the arms and budget to inspect any other pair. Differences here are in plain macro-F1 (0.004 = 0.4 points).

In [ ]:
an.paired_bootstrap(runs, "A5", "A4", budget=10, split="test", metric="macro_f1", n=N_BOOT, rng_seed=RNG_SEED)

## 8. Figure: score against label budget
One line per arm: mean test macro-F1 against the share of labeled data (log axis), with error bars of one sd over seeds. It shows where the arms differ most (expected at small budgets) and where they converge.

In [ ]:
fig = an.plot_f1_vs_budget(runs, OUT_DIR, split="test", metric="macro_f1")

## 9. Figure: GAN training curves
For the two GAN arms (A4, A5), the discriminator's probability of calling a text "fake", for real texts and for generated ones, over the training steps (mean over seeds, band = min to max). If the generated line stays far above the real line the discriminator wins easily; if the two meet, the generator is fooling it. This is evidence for *why* the GAN part does or does not help.

In [ ]:
fig = an.plot_gan_curves(runs, OUT_DIR)

## 10. iSarcasmEval table
The same models scored on a different sarcasm dataset (iSarcasmEval test, no retraining): macro-F1, F1 of the sarcastic class (that benchmark's official metric) and ROC-AUC, mean ± sd over seeds. It checks whether anything carries over outside news headlines.

In [ ]:
isarc = an.isarc_table(runs)
an.save_table(isarc, OUT_DIR, "isarc_test")
isarc

## 11. Where the files went
Lists everything written to the output folder (CSV and Markdown tables, PDF and PNG figures). On Kaggle the folder is also zipped to `results_analysis.zip`: download it from the Output tab.

In [ ]:
if ON_KAGGLE:
    import shutil
    shutil.make_archive("/kaggle/working/results_analysis", "zip", OUT_DIR)
    print("zipped to /kaggle/working/results_analysis.zip")
sorted(p.name for p in Path(OUT_DIR).iterdir())